<a href="https://colab.research.google.com/github/seamusrobertmurphy/TREES-demo-repository/blob/main/Winrock_Ecuador_Nesting_Deliverable_4_notes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Environment setup

In [ ]:
!pip install leafmap geemap==0.16.4 geopandas numpy session_info
import ee, json, geemap, ipyleaflet, os, numpy, backports, session_info
from google.colab import drive
from google.colab import files
!drive.mount('/content/drive')

In [ ]:
# Check Runtime GPU
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0: print('Not connected to a GPU')
else: print(gpu_info)

# Check Runtime-Allocated RAM
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
if ram_gb < 20: print('Not using a high-RAM runtime')
else: print('You are using a high-RAM runtime!')

/bin/bash: line 1: nvidia-smi: command not found
Your runtime has 359.2 gigabytes of available RAM

You are using a high-RAM runtime!


##### *Activate Earth Engine*

In [ ]:
#!ee.Authenticate() # deprecated in certain Colab environments
!earthengine authenticate
ee.Initialize(project = "murphys-deforisk")

Authenticate: Limited support in Colab. Use ee.Authenticate() or --auth_mode=notebook instead.
Authenticate: Credentials already exist.  Use --force to refresh.


### **Jurisidictional boundary: AOI**

In [ ]:
country = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(
    ee.Filter.equals("ADM0_NAME", "Ecuador"))
state = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(
    ee.Filter.equals("ADM0_NAME", "Ecuador"))
state_list = country.aggregate_array('ADM1_NAME').distinct().getInfo()
state_list

centroid_ee = country.first().geometry().centroid()

In [ ]:
red = {"color": "red", "width": 1, "lineType": "solid", "fillColor": "00000000"}
white = {"color": "white", "width": 1, "lineType": "solid", "fillColor": "00000000"}
country_label = ee.FeatureCollection([ee.Feature(
    country.geometry().centroid(), {'country_name': country.first().get("ADM0_NAME").getInfo()})])

Map = geemap.Map()
#Map.centerObject(country, 6)
Map.centerObject(centroid_ee, 6)
Map.add_basemap('Esri.WorldImagery')
Map.addLayer(country.style(**white), {}, "Country")
Map.addLayer(state.style(**red), {}, "States")
Map.add_labels(state,"ADM1_NAME",font_size="6pt",font_color="white",font_family="arial") #,font_weight="bold",)
Map.add_labels(country_label, "country_name", font_size="12pt", font_color="white", font_family="arial",)
Map

Map(center=[-0.6564715832209803, -77.82173076923884], controls=(WidgetControl(options=['position', 'transparen…

### **Sentinel collections & Cloud Score Plus masking**

This Cloud Score Plus dataset is a useful option in this AOI due to high cloud cover. This requires the quality band filter from the 'GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED' dataset  and  the function `qualityMosaic(QA_BAND)` to back-fill missing pixels according to a cloud percentage `cs` or cloud probability ranking `cs_dsf`.


Recommended pipeline [here](https://www.google.com/url?q=https%3A%2F%2Fcode.earthengine.google.com%2F%3FscriptPath%3DExamples%253ACloud%2520Masking%252FSentinel2%2520Cloud%2520And%2520Shadow)

In [ ]:
# Cloud Score+ image collection for Sentinel-2: Best for filling gaps
cs_plus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
QA_BAND = 'cs'
CLEAR_THRESHOLD = 0.40 # Values between 0.50 and 0.65 are common.

# Replace the median composite with cloud ranking.
composite_2025 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate('2024-05-01', '2025-05-01') \
    .filterBounds(country) \
    .linkCollection(cs_plus, [QA_BAND]) \
    .map(lambda img: img.updateMask(img.select(QA_BAND).gte(CLEAR_THRESHOLD))) \
    .qualityMosaic(QA_BAND).clip(country).toFloat()

# Visualization.
Map = geemap.Map()
Map.centerObject(centroid_ee, 6)
Map.addLayer(composite_2025, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 2500}, 'RGB 2025')
Map


### **Sentinel collections & default masking**

 This example uses the Sentinel-2 QA band to cloud mask that is less strictly filtered, so collection is also pre-filtered by CLOUDY_PIXEL_PERCENTAGE.



In [ ]:
def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
           qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000) \
        .select("B.*") \
        .copyProperties(image, ["system:time_start"])

collection_2017 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
    .filterDate('2017-07-01', '2018-07-01') \
    .filterBounds(country) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60)) \
    .map(maskS2clouds)

collection_2021 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
    .filterDate('2021-07-01', '2022-07-01') \
    .filterBounds(country) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60)) \
    .map(maskS2clouds)

collection_2025 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
    .filterDate('2024-07-01', '2025-07-01') \
    .filterBounds(country) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60)) \
    .map(maskS2clouds)

composite_2017 = collection_2017.median().clip(country).toFloat()
composite_2021 = collection_2021.median().clip(country).toFloat()
composite_2025 = collection_2025.median().clip(country).toFloat()

Map = geemap.Map()
Map.centerObject(country, 5)
Map.addLayer(composite_2017, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'RGB 2017')
Map.addLayer(composite_2021, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'RGB 2021')
Map.addLayer(composite_2025, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'RGB 2025')
Map

### **Landsat collections & processing**

The time series was derived from Landsat's Collection-2 Level-2 Tier-1 of Surface Reflectance imagery, as described in earth engine catalog here: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2.
This includes rasters preprocessed with atmospheric corrections using the LaSRC and includes a cloud, shadow, water and snow mask produced using CFMASK, as well as a per-pixel saturation mask.

In [ ]:
# derive masking, scaling, and ndvi function
def maskL8sr(image):
    qaMask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturationMask = image.select('QA_RADSAT').eq(0)
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI').toFloat()
    image = image.addBands(opticalBands, None, True) \
                 .addBands(thermalBands, None, True) \
                 .addBands(ndvi)
    return image.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'NDVI'],
        ['BLUE', 'GREEN', 'RED', 'NIR08', 'SWIR16', 'SWIR22', 'NDVI']
    ).updateMask(qaMask).updateMask(saturationMask)

# create collections for 2014 and 2024
collection_2014 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
                    .filterDate('2014-01-01', '2014-12-31') \
                    .filterBounds(state) \
                    .map(maskL8sr)

collection_2019 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
                    .filterDate('2019-01-01', '2019-12-31') \
                    .filterBounds(state) \
                    .map(maskL8sr)

collection_2024 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
                    .filterDate('2024-01-01', '2024-12-31') \
                    .filterBounds(state) \
                    .map(maskL8sr)

# median composites for 2014 and 2024
composite_2014 = collection_2014.select(['BLUE', 'GREEN', 'RED', 'NIR08', 'SWIR16', 'SWIR22', 'NDVI']).median().clip(state).toFloat()
composite_2019 = collection_2019.select(['BLUE', 'GREEN', 'RED', 'NIR08', 'SWIR16', 'SWIR22', 'NDVI']).median().clip(state).toFloat()
composite_2024 = collection_2024.select(['BLUE', 'GREEN', 'RED', 'NIR08', 'SWIR16', 'SWIR22', 'NDVI']).median().clip(state).toFloat()


# visualization
ndviVis = {'min': 0.2, 'max': 0.8, 'palette': ['red', 'yellow', 'green']}
rgbVis = {'bands': ['RED', 'GREEN', 'BLUE'],'min': 0, 'max': 0.3, 'gamma': 1.4}
Map = geemap.Map()
Map.centerObject(country, 5)
#Map.addLayer(composite_2014.select('NDVI'), ndviVis, 'NDVI 2014')
#Map.addLayer(composite_2019.select('NDVI'), ndviVis, 'NDVI 2019')
#Map.addLayer(composite_2024.select('NDVI'), ndviVis, 'NDVI 2024')
#Map.addLayer(composite_2014.select(['RED', 'GREEN', 'BLUE']), rgbVis, 'RGB 2014')
#Map.addLayer(composite_2019.select(['RED', 'GREEN', 'BLUE']), rgbVis, 'RGB 2019')
Map.addLayer(composite_2024.select(['RED', 'GREEN', 'BLUE']), rgbVis, 'RGB 2024')
Map.addLayer(state, {}, 'Area of Interest')
Map.addLayerControl()
Map

Map(center=[-1.432800372103166, -78.76612016462705], controls=(WidgetControl(options=['position', 'transparent…

### GLANCE Training Dataset

Information of training sample and associated datasets available here: https://gee-community-catalog.org/projects/glance_training/
Published paper describing methods to develop the training sample available here: https://www.nature.com/articles/s41597-023-02798-5

In [ ]:
glance_training = ee.FeatureCollection("projects/sat-io/open-datasets/GLANCE/GLANCE_TRAINING_DATA_V1")
#print(glance_training.first().getInfo())

Map = geemap.Map()
Map.centerObject(country, 6)
Map.add_basemap('Esri.WorldImagery')
Map.addLayer(glance_training.style(**red), {}, 'GLANCE Training Data')
Map


Map(center=[-1.432800372103166, -78.76612016462705], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
glance_training_list = glance_training.aggregate_array('Glance_Class_ID_level1').distinct().getInfo()
glance_training_list


[5, 1, 3, 4, 7, 6, 2]